### Step 1: Environment Setup and Directory Initialization
This cell prepares the Google Colab environment for the data engineering pipeline.
First, it installs the necessary libraries: `kagglehub` and `roboflow` for dataset acquisition, and `albumentations` for advanced image augmentations. Then, it imports the required modules and constructs the master directory structure.

YOLO requires a very specific directory format to train properly: separate `images` and `labels` folders, each containing `train` and `val` subdirectories. This cell ensures that our unified workspace (`/content/GUARDIAN_Dataset/unified_yolo`) is built exactly to this standard before any data is processed.

In [1]:
# Cell 1: Installations and Imports
!apt-get update && apt-get install -y libglib2.0-0 libsm6 libxext6 libxrender-dev libgl1

!pip install -q kagglehub roboflow albumentations opencv-python python-dotenv

from dotenv import load_dotenv
import os
import cv2
import shutil
import kagglehub
from roboflow import Roboflow
import albumentations as A
from glob import glob
from pathlib import Path

# Define the master directory for the unified dataset
BASE_DIR = '/content/GUARDIAN_Dataset'
UNIFIED_DIR = os.path.join(BASE_DIR, 'unified_yolo')

# Create standard YOLO directory structure
for split in ['train', 'val']:
    os.makedirs(os.path.join(UNIFIED_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(UNIFIED_DIR, 'labels', split), exist_ok=True)

print(f"Unified dataset structure created at: {UNIFIED_DIR}")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1 is already the newest version (1.4.0-1).
libsm6 is already the newest version (2:1.2.3-1build2).
libxext6 is already the newest version (2:1.3.4-1build1).
libxrender-dev is already the newest version (1:0.9.10-1build4).
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
0 upgraded, 0 newly installed, 0 to remove and 137 not upgraded.

[notice] A new release of pip is available: 25.2 

### Step 2: Data Acquisition from Kaggle and Roboflow
To train the GUARDIAN object detection model robustly, we need diverse data containing weapons and suspicious behaviors. This cell connects to two major repositories:
1. **Kaggle:** Downloads a specialized weapon detection dataset.
2. **Roboflow:** Uses the Roboflow API to download two specific datasets (suspicious movements and gun detection from webcam views) directly in the YOLOv8 format.

In [2]:
# Cell 2: Download Datasets and load env variables
raw_datasets = {}
load_dotenv(dotenv_path="config.env")

print("Downloading Kaggle Dataset: weapon-detection...")
kaggle_path = kagglehub.dataset_download("mehmetcubukcu/weapon-detection")
raw_datasets['kaggle_weapon'] = kaggle_path
print(f"Kaggle dataset downloaded to: {kaggle_path}\n")

try:
    print("Initializing Roboflow...")
    api_key = os.getenv("ROBOFLOW_API_KEY")
    
    if not api_key:
        raise ValueError("ROBOFLOW_API_KEY is missing from environment variables/ .env file")

    # Connecting to API
    rf = Roboflow(api_key=api_key)
    print("Roboflow initialized successfully.")
    
    print("Downloading Roboflow: Suspicious movement...")
    project_sm = rf.workspace("suspicious-movement").project("suspicious-detection")
    dataset_sm = project_sm.version(1).download("yolov8")
    raw_datasets['robo_suspicious'] = dataset_sm.location
    
    print("\nDownloading Roboflow: Gun with webcam views...")
    project_gw = rf.workspace("jiraphat-stgwm").project("gun-with-webcam-views")
    dataset_gw = project_gw.version(3).download("yolov8")
    raw_datasets['robo_gun'] = dataset_gw.location
    
    print("\nAll datasets downloaded successfully!")

except ValueError as ve:
    print(f"Configuration Error: {ve}")
except Exception as e:
    print(f"An unexpected error occurred during Roboflow initialization: {e}")

Kaggle dataset downloaded to: /root/.cache/kagglehub/datasets/mehmetcubukcu/weapon-detection/versions/1

Initializing Roboflow...
Roboflow initialized successfully.
loading Roboflow workspace...
loading Roboflow project...

loading Roboflow workspace...
loading Roboflow project...

All datasets downloaded successfully!


### Step 2.5: Dataset Classes Verification
Before unifying the datasets, this cell reads the `.yaml` configuration files from each downloaded dataset (Roboflow and Kaggle) to identify their specific class mappings and IDs. This step is crucial for understanding how to map the various object IDs (like 'pistol', 'knife', 'suspicious') into our unified system.

In [3]:
import yaml
import os
from glob import glob

# Define the paths for all datasets
dataset_paths = {
    "Suspicious Movement (Roboflow)": dataset_sm.location,
    "Guns Webcam (Roboflow)": dataset_gw.location,
    "Weapon Detection (Kaggle)": kaggle_path
}

for name, directory in dataset_paths.items():
    yaml_files = glob(os.path.join(directory, '**', '*.yaml'), recursive=True)

    if yaml_files:
        yaml_path = yaml_files[0] # Select the first YAML file found
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
            print(f"--- Classes for {name} ---")

            names = data.get('names', [])

            if isinstance(names, dict):
                for index, class_name in names.items():
                    print(f"ID {index} = {class_name}")
            elif isinstance(names, list):
                for index, class_name in enumerate(names):
                    print(f"ID {index} = {class_name}")
            print("\n")
    else:
        print(f"--- Classes for {name} ---")
        print(f"⚠️ Could not find a .yaml file anywhere inside {directory}\n")

--- Classes for Suspicious Movement (Roboflow) ---
ID 0 = suspicious-suspect
ID 1 = victim
ID 2 = weapon


--- Classes for Guns Webcam (Roboflow) ---
ID 0 = -
ID 1 = Guns


--- Classes for Weapon Detection (Kaggle) ---
ID 0 = pistol
ID 1 = smartphone
ID 2 = knife
ID 3 = monedero
ID 4 = billete
ID 5 = tarjeta




### Step 3: Dataset Unification Engine
Because datasets from different sources often have varied internal folder hierarchies, this cell standardizes them.

The `unify_datasets` function scans through the downloaded raw datasets recursively. It looks for image files and strictly searches for their corresponding `.txt` YOLO bounding box label files. If a matched pair is found, it copies both files into our newly created unified `train` directories. It also prefixes the filenames with the dataset source name to prevent overwriting files that happen to share the same name (e.g., `image_01.jpg`).

In [4]:
# Cell 3: Unification Engine
import os
import shutil
from glob import glob

# STEP 1: Define the unified class mapping dictionary.
# Format -> 'dataset_name': {old_id: new_id}
CLASS_MAP = {
    # Suspicious Movement dataset: Only class 2 is 'weapon'
    'suspicious_movement': {
        2: 0  # Map original ID 2 ('weapon') to our new ID 0 ('Gun')
    },

    # Guns Webcam dataset: Only class 1 is 'Guns'
    'roboflow_guns': {
        1: 0  # Map original ID 1 ('Guns') to our new ID 0 ('Gun')
    },

    # Kaggle dataset: Class 0 is 'pistol', Class 2 is 'knife'
    'kaggle_weapon': {
        0: 0, # 'pistol' remains ID 0 ('Gun')
        2: 1  # 'knife' becomes ID 1 ('Knife')
    }
}

def unify_datasets_with_mapping(source_dirs, target_dir, split='train'):
    """
    Iterates over multiple datasets, copies images, and rewrites YOLO label files (.txt).
    It filters out unwanted classes and updates class IDs to ensure uniformity.
    """
    target_images = os.path.join(target_dir, 'images', split)
    target_labels = os.path.join(target_dir, 'labels', split)
    total_copied = 0

    for name, source_dir in source_dirs.items():
        print(f"Processing dataset: {name}...")

        # Find all .jpg and .png images recursively in the source directory
        image_files = glob(os.path.join(source_dir, '**', '*.jpg'), recursive=True) + \
                      glob(os.path.join(source_dir, '**', '*.png'), recursive=True)

        for img_path in image_files:
            # Extract the filename without the extension
            base_name = os.path.splitext(os.path.basename(img_path))[0]

            # Construct the path to the corresponding .txt label file
            txt_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')

            # Proceed only if the label file actually exists
            if os.path.exists(txt_path):
                # Add dataset name as a prefix to prevent overwriting images with the same name
                new_file_name = f"{name}_{base_name}"

                # Read the original label file
                with open(txt_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                # Process each bounding box row in the label file
                for line in lines:
                    parts = line.strip().split()
                    if not parts:
                        continue # Skip empty lines

                    old_id = int(parts[0])

                    # FILTERING MAGIC:
                    # Only translate and keep the bounding box if the old_id is in our mapping dictionary!
                    if name in CLASS_MAP and old_id in CLASS_MAP[name]:
                        new_id = CLASS_MAP[name][old_id]
                        # Reconstruct the YOLO format row with the NEW class ID
                        new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")

                # Only copy files if we successfully kept at least one relevant label (Gun/Knife)
                if new_lines:
                    # 1. Copy the image to the unified directory
                    shutil.copy(img_path, os.path.join(target_images, f"{new_file_name}.jpg"))

                    # 2. Write the newly translated and filtered bounding boxes to the unified labels directory
                    with open(os.path.join(target_labels, f"{new_file_name}.txt"), 'w') as f:
                        f.writelines(new_lines)

                    total_copied += 1

    print(f"Success! Total relevant images (containing Guns/Knives) copied and mapped: {total_copied}")

# Execution: Call the function with your actual datasets
# NOTE: Using the keys 'robo_gun' and 'robo_suspicious' based on your Cell 2 variables
unify_datasets_with_mapping({
    'kaggle_weapon': raw_datasets.get('kaggle_weapon', ''),
    'roboflow_guns': raw_datasets.get('robo_gun', ''),
    'suspicious_movement': raw_datasets.get('robo_suspicious', '')
}, UNIFIED_DIR, split='train')

Processing dataset: kaggle_weapon...
Processing dataset: roboflow_guns...
Processing dataset: suspicious_movement...
Success! Total relevant images (containing Guns/Knives) copied and mapped: 6526


### Step 4: Defining the CCTV-Specific Augmentation Pipeline
To ensure the GUARDIAN system performs well on low-end hardware and cheap CCTV infrastructure, we must simulate those poor environmental conditions during training.

This cell utilizes `albumentations` to define a transformation pipeline that applies Motion Blur, Gaussian Noise, Perspective Distortion, and Image Compression.

**Crucial Component:** The `A.BboxParams(format='yolo')` argument is heavily utilized here. It ensures that when the image undergoes geometric transformations (like perspective shifts), the coordinates of the YOLO bounding boxes are mathematically recalculated to stay perfectly aligned with the target object. It also drops boxes that fall out of the frame entirely (`min_visibility`).

In [5]:
# Cell 4: Augmentation Pipeline Initialization
# Define the augmentation pipeline tailored for CCTV constraints
cctv_transform = A.Compose([
    # Simulates camera capturing fast-moving subjects (e.g., someone pulling a weapon)
    A.MotionBlur(blur_limit=15, p=0.4),

    # Simulates low-light sensor noise typical in cheap CCTV cameras
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),

    # Simulates the high-angle/corner placement of surveillance cameras
    A.Perspective(scale=(0.05, 0.1), p=0.3),

    # Simulates DVR/NVR heavy video compression
    A.ImageCompression(quality_lower=30, quality_upper=70, p=0.5),

], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.2  # Drops boxes if the augmentation pushes them mostly out of frame
))

print("CCTV Augmentation pipeline with YOLO BBox support initialized.")

CCTV Augmentation pipeline with YOLO BBox support initialized.


Argument(s) 'var_limit' are not valid for transform GaussNoise
Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression


### Step 5: Augmentation Execution & Generating Synthetic CCTV Samples
This final data preparation step applies the defined CCTV pipeline to our unified dataset.

The `augment_yolo_dataset` function iterates through every image in the unified `train` directory. For each image, it opens the corresponding YOLO `.txt` file, parses the coordinates, and feeds both the image and the coordinates into the `albumentations` pipeline. Finally, it saves a new synthetic CCTV image alongside a newly generated `.txt` file with the correctly adjusted bounding box coordinates. This effectively doubles the size and robustness of our training dataset.

In [6]:
# Cell 5: Augmentation Execution Engine
def augment_yolo_dataset(target_dir, transform, split='train'):
    images_dir = os.path.join(target_dir, 'images', split)
    labels_dir = os.path.join(target_dir, 'labels', split)

    image_files = glob(os.path.join(images_dir, '*.jpg'))
    print(f"Starting augmentation on {len(image_files)} images...")

    success_count = 0

    for img_path in image_files:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(labels_dir, f"{base_name}.txt")

        if not os.path.exists(label_path):
            continue

        # 1. Read the image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Parse the YOLO label file
        bboxes = []
        class_labels = []
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(float(parts[0]))
                    # x_center, y_center, width, height
                    coords = list(map(float, parts[1:]))
                    bboxes.append(coords)
                    class_labels.append(class_id)

        if len(bboxes) == 0:
            continue # Skip images with empty annotations

        # 3. Apply the augmentations
        try:
            transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
            transformed_image = transformed['image']
            transformed_bboxes = transformed['bboxes']
            transformed_classes = transformed['class_labels']

            # Skip if augmentation removed all valid boxes
            if len(transformed_bboxes) == 0:
                continue

            # 4. Save the augmented image
            aug_base_name = f"aug_cctv_{base_name}"
            aug_img_path = os.path.join(images_dir, f"{aug_base_name}.jpg")

            transformed_image_bgr = cv2.cvtColor(transformed_image, cv2.COLOR_RGB2BGR)
            cv2.imwrite(aug_img_path, transformed_image_bgr)

            # 5. Save the augmented YOLO labels
            aug_label_path = os.path.join(labels_dir, f"{aug_base_name}.txt")
            with open(aug_label_path, 'w') as f:
                for bbox, cls_id in zip(transformed_bboxes, transformed_classes):
                    f.write(f"{cls_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")

            success_count += 1

        except ValueError as e:
            # Albumentations throws ValueErrors if coordinate limits are violated
            pass

    print(f"Augmentation complete! Generated {success_count} new CCTV-simulated samples.")

# Run the augmentation engine
augment_yolo_dataset(UNIFIED_DIR, cctv_transform, split='train')

Starting augmentation on 11957 images...
Augmentation complete! Generated 11744 new CCTV-simulated samples.


### Step 6: Dataset Configuration (`data.yaml`)
YOLOv8 requires a `data.yaml` configuration file to understand where the dataset is located and what classes it needs to predict.

This cell dynamically generates this YAML file directly in our unified dataset directory. It defines the paths to our `train` and `val` sets and explicitly lists the class names we unified in the previous step (e.g., 'gun', 'knife', 'suspicious'). This configuration ensures the model maps the numerical class IDs in our `.txt` files to human-readable labels.

In [7]:
# Cell 6: Generate YOLO Configuration File
import yaml
import os

# Define paths
UNIFIED_DIR = '/content/GUARDIAN_Dataset/unified_yolo'
YAML_PATH = os.path.join(UNIFIED_DIR, 'data.yaml')

# Define the dataset configuration focusing ONLY on weapons and knives
# nc = number of classes (we only have 2 now)
# names = list of class names in the exact order of their IDs (0: Gun, 1: Knife)
guardian_data_config = {
    'path': UNIFIED_DIR,           # dataset root dir
    'train': 'images/train',       # train images (relative to 'path')
    'val': 'images/val',           # val images (relative to 'path')
    'nc': 2,                       # We now have exactly 2 classes
    'names': ['Gun', 'Knife']      # ID 0 = Gun, ID 1 = Knife
}

# Write the configuration to the data.yaml file
with open(YAML_PATH, 'w') as f:
    yaml.dump(guardian_data_config, f, default_flow_style=False)

print(f"data.yaml successfully generated at {YAML_PATH}")
print("Classes configured: 0: Gun, 1: Knife")

data.yaml successfully generated at /content/GUARDIAN_Dataset/unified_yolo/data.yaml
Classes configured: 0: Gun, 1: Knife


### Step 7: Train/Validation Split Fix
YOLOv8 requires both a training set (to learn from) and a validation set (to test itself at the end of every training epoch).
During our data unification step, all processed images and labels were placed into the `train` folder, leaving the `val` folder empty. This is what caused the initial `FileNotFoundError`.
This cell resolves the issue by dynamically splitting our dataset. It randomly selects 15% of the images and their corresponding YOLO `.txt` labels from the `train` directory and moves them to the `val` directory, adhering to standard Deep Learning practices.

In [8]:
# Cell 7: Train/Validation Split Fix
import os
import random
import shutil
from glob import glob

# Define paths
TRAIN_IMG_DIR = '/content/GUARDIAN_Dataset/unified_yolo/images/train'
TRAIN_LBL_DIR = '/content/GUARDIAN_Dataset/unified_yolo/labels/train'
VAL_IMG_DIR = '/content/GUARDIAN_Dataset/unified_yolo/images/val'
VAL_LBL_DIR = '/content/GUARDIAN_Dataset/unified_yolo/labels/val'

# Get all images currently in train
all_images = glob(os.path.join(TRAIN_IMG_DIR, '*.jpg'))

# Calculate how many to move (15% for validation)
val_split_ratio = 0.15
num_val = int(len(all_images) * val_split_ratio)

print(f"Total images currently in train: {len(all_images)}")
print(f"Moving {num_val} random images to validation set...")

# Randomly select images to move (set seed for reproducibility)
random.seed(42)
val_images = random.sample(all_images, num_val)

# Move the files
for img_path in val_images:
    base_name = os.path.basename(img_path)
    txt_name = base_name.replace('.jpg', '.txt')

    lbl_path = os.path.join(TRAIN_LBL_DIR, txt_name)

    # Move image
    shutil.move(img_path, os.path.join(VAL_IMG_DIR, base_name))

    # Move label if it exists
    if os.path.exists(lbl_path):
        shutil.move(lbl_path, os.path.join(VAL_LBL_DIR, txt_name))

print("Split complete! Ready for YOLO training.")
print(f"Final Train size: {len(os.listdir(TRAIN_IMG_DIR))} images")
print(f"Final Val size: {len(os.listdir(VAL_IMG_DIR))} images")

Total images currently in train: 18306
Moving 2745 random images to validation set...
Split complete! Ready for YOLO training.
Final Train size: 15561 images
Final Val size: 4400 images


### Step 8: Baseline Model Initialization & Sanity Check Training
In this phase, we initialize our Deep Learning pipeline using the `ultralytics` package.

We use `yolov8m.pt` (YOLOv8 Medium) to establish a strong baseline model. We run a "Sanity Check" training loop for a short number of epochs (15). The goal here is not to achieve state-of-the-art accuracy immediately, but to verify that:
1. The data pipeline is feeding correctly without errors.
2. The bounding boxes are properly aligned.
3. The training loss converges (goes down) and we get an initial Mean Average Precision (mAP) score.

In [9]:
# Cell 8: Sanity Check Training Loop
!pip install -q ultralytics
from ultralytics import YOLO

# 1. Initialize the Medium baseline model
print("Initializing YOLOv8 Medium baseline model...")
model = YOLO('yolov8m.pt')

# 2. Run the Sanity Check Training
print("Starting Sanity Check Training...")
# Note: Ensure data.yaml path matches what was created in Step 6
YAML_PATH = '/content/GUARDIAN_Dataset/unified_yolo/data.yaml'

results = model.train(
    data=YAML_PATH,
    epochs=15,             # Short run just to verify convergence
    imgsz=640,             # Standard resolution for CCTV
    batch=16,              # Batch size optimized for Colab's T4 GPU
    device=0,              # Force use of GPU
    project='GUARDIAN',    # Master folder for runs
    name='sanity_check_01',# Name of this specific experiment
    exist_ok=True,         # Overwrite if we restart the cell
    patience=5             # Early stopping if loss stops improving
)

print("Sanity check complete! Validation metrics (mAP) have been logged.")


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Initializing YOLOv8 Medium baseline model...
Starting Sanity Check Training...
Ultralytics 8.4.47 🚀 Python-3.11.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 20556MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/GUARDIAN_Dataset/unified_yolo/data.yaml, degrees=0.0, determinis

### Step 9: Custom Video Upload and Threat Inference
Now we transition from testing on static dataset images to real-world video testing.
This cell allows you to upload a custom CCTV `.mp4` or `.avi` file.

To solve the "money/phone" noise issue efficiently, we use the `classes` parameter. Based on our `data.yaml`, classes **0 and 1** represent Guns and Knives. By passing `classes=[0, 1]`, the model will ignore any other objects it might have learned and only track these specific relevant threats.

In [15]:
import os
import shutil
from ultralytics import YOLO

# 1. Provide the exact name of the video file you uploaded to JupyterLab
# Example: 'my_phone_video.mp4'
video_file_name = 'test_video.mp4' 

# Safety check
if not os.path.exists(video_file_name):
    raise FileNotFoundError(f"❌ Could not find {video_file_name}. Make sure it is uploaded to JupyterLab!")

# 2. Clean up previous test results
testing_dir = 'runs/detect/Custom_Testing'
if os.path.exists(testing_dir):
    shutil.rmtree(testing_dir)

# 3. Define the path to our custom-trained GUARDIAN model weights
best_weights_path = 'runs/detect/GUARDIAN/sanity_check_01/weights/best.pt'
if not os.path.exists(best_weights_path):
    best_weights_path = 'runs/detect/GUARDIAN/sanity_check_01/weights/last.pt'
    print(f"⚠️ 'best.pt' not found. Falling back to {best_weights_path}")

# Initialize the YOLO model
guardian_model = YOLO(best_weights_path)

# 4. Execute the model TRACKING on the uploaded video file
print(f"\n🔍 Analyzing and Tracking file: {video_file_name}...")

# USING model.track() FOR CONTINUOUS OBJECT TRACKING
results = guardian_model.track(
    source=video_file_name, 
    conf=0.25,            
    classes=[0, 1],       
    persist=True,         
    tracker="botsort.yaml",
    save=True,            # This saves the final tracked video to the folder!
    project='Custom_Testing',
    name='threat_detection',
    exist_ok=True
)

print("-" * 50)
print(f"✅ Tracking complete! The file is ready for Cell 10 to display.")
print("-" * 50)


🔍 Connecting to camera and tracking stream at: https://bannister-urgency-cesspool.ngrok-free.dev/video...
⚠️ Note: If a window opens, you must click on it and press 'q' to stop tracking.



ConnectionError: 1/1: https://bannister-urgency-cesspool.ngrok-free.dev/video... Failed to open https://bannister-urgency-cesspool.ngrok-free.dev/video

### Step 10: Process Output Video and Download
YOLOv8 outputs the processed video into the `Custom_Testing` directory.
Since raw detection videos (often in `.avi`) might not play directly in modern browsers, this cell converts the result into a high-quality, web-ready `.mp4` using `ffmpeg`.

It then provides a direct download button for the final annotated video showing the detected threats and persistent ByteTrack IDs.

In [ ]:
# Cell 10: Display Output (Image or Video) - JupyterLab Version
import os
import glob
import subprocess
from IPython.display import Image, HTML, display
from base64 import b64encode

# 1. Clean up OLD display files to prevent showing artifacts from previous runs
# Removed Colab-specific '/content/' paths
FINAL_OUTPUT = 'guardian_threat_report.mp4'
PREVIEW_PATH = 'preview.mp4'

for old_file in [FINAL_OUTPUT, PREVIEW_PATH]:
    if os.path.exists(old_file):
        os.remove(old_file)
        print(f"🗑️ Deleted old artifact: {old_file}")

# 2. Smartly find the LATEST YOLO output directory
# (YOLO creates incremental folders like threat_detection, threat_detection2, etc.)
testing_base = 'runs/detect/Custom_Testing'

# Sort directories by modification time to always grab the newest one
# We use a try-except block just in case the directory doesn't exist yet
try:
    subdirs = sorted(glob.glob(os.path.join(testing_base, 'threat_detection*')), key=os.path.getmtime, reverse=True)
except FileNotFoundError:
    subdirs = []

if not subdirs:
    print("⚠️ No result directories found. Did Cell 9 execute successfully and save an output?")
else:
    RESULT_DIR = subdirs[0] # Select the most recently created folder
    print(f"📂 Reading results from the latest directory: {RESULT_DIR}")

    # 3. Scan the directory for generated images or videos
    image_results = glob.glob(os.path.join(RESULT_DIR, '*.jpg')) + glob.glob(os.path.join(RESULT_DIR, '*.png'))
    video_results = glob.glob(os.path.join(RESULT_DIR, '*.avi')) + glob.glob(os.path.join(RESULT_DIR, '*.mp4'))

    # --- Handle Image Output ---
    if image_results:
        output_img = image_results[0]
        print(f"🖼️ Result is an image ({os.path.basename(output_img)}). Displaying detection:")
        display(Image(filename=output_img, width=800))

        # JupyterLab-friendly HTML Download Button
        print("\n" + "="*30)
        download_button = f'''
        <a href="{output_img}" download="{os.path.basename(output_img)}" target="_blank" 
           style="display: inline-block; padding: 10px 20px; background-color: #28a745; color: white; 
                  text-decoration: none; border-radius: 5px; font-family: sans-serif; font-weight: bold;">
           📥 Download Image
        </a>
        '''
        display(HTML(download_button))

    # --- Handle Video Output ---
    elif video_results:
        raw_video = video_results[0]
        print(f"🎬 Result is a video ({os.path.basename(raw_video)}). Converting format for browser display...")

        # Convert to web-compatible MP4 format
        cmd = f'ffmpeg -y -i "{raw_video}" -vcodec libx264 -pix_fmt yuv420p "{FINAL_OUTPUT}" -loglevel quiet'
        subprocess.run(cmd, shell=True)

        if os.path.exists(FINAL_OUTPUT):
            # Create a short 10-second preview strictly for the notebook UI rendering
            subprocess.run(f'ffmpeg -y -i "{FINAL_OUTPUT}" -t 10 -c copy "{PREVIEW_PATH}" -loglevel quiet', shell=True)

            mp4_data = open(PREVIEW_PATH, 'rb').read()
            data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

            print("\n✅ Displaying a preview of the new detection:")
            display(HTML(f"""
                <video width=800 controls autoplay loop>
                      <source src="{data_url}" type="video/mp4">
                </video>
            """))

            # JupyterLab-friendly HTML Download Button for the full length video
            print("\n" + "="*30)
            download_button = f'''
            <a href="{FINAL_OUTPUT}" download="guardian_threat_report.mp4" target="_blank" 
               style="display: inline-block; padding: 10px 20px; background-color: #28a745; color: white; 
                      text-decoration: none; border-radius: 5px; font-family: sans-serif; font-weight: bold;">
               📥 Download Full Video
            </a>
            '''
            display(HTML(download_button))
        else:
            print("⚠️ Video conversion failed. Please check your ffmpeg installation.")
    else:
        print("⚠️ No image or video files found in the results directory.")

### Step 11: Visualize Training Metrics (Loss & Accuracy)
After seeing the model in action, we analyze the learning curves to ensure the model is properly optimized.
YOLOv8 automatically logs all training metrics into a single image file (`results.png`).

We are looking for:
1. **Loss Curves:** Should consistently decrease.
2. **mAP Metrics:** Should consistently increase, showing better detection accuracy.

In [ ]:
# Cell 11: Visualize Training Results - JupyterLab Version
import os
from IPython.display import Image, HTML, display

# 1. Updated path using relative paths for JupyterLab (Removed '/content/')
results_img_path = 'runs/detect/GUARDIAN/sanity_check_01/results.png'

if os.path.exists(results_img_path):
    print("📈 Displaying Training Results (Loss and mAP Metrics):")
    
    # Display the image inside the Jupyter Notebook cell
    display(Image(filename=results_img_path, width=1200))

    # 2. JupyterLab-friendly HTML Download Button
    print("\n" + "="*30)
    download_button = f'''
    <a href="{results_img_path}" download="guardian_training_results.png" target="_blank" 
       style="display: inline-block; padding: 10px 20px; background-color: #007bff; color: white; 
              text-decoration: none; border-radius: 5px; font-family: sans-serif; font-weight: bold;">
       📊 Download Results Graph
    </a>
    '''
    display(HTML(download_button))
else:
    print(f"⚠️ Results image not found at '{results_img_path}'.")
    print("Ensure that the training step (Cell 8) completed successfully and saved to the correct directory.")